Enhancing Reasoning Capability in LLMs
===


**Created by Dr Chao Shu (chao.shu@qmul.ac.uk)**

Submitted By

**Name**: Luoyi Zhu

**QMUL ID**: 241119954

**BUPT ID**: 2024213652

# Introduction
---

In this project, you are expected to comprehensively apply the reasoning and inference techniques you have learned in L01, L2 and L05 (L03 & L06 were covered in the Coding Challenge) in this module to unlock the reasoning potential of a small-scale non-reasoning model. You will build an end-to-end pipeline to fine-tune the model to improve its reasoning capability and benchmark its reasoning performance on the GSM8k dataset. This notebook will guide to you to:

1. **Load** a pre-trained instruct model and attach a LoRA adapter for efficient fine-tuning.
2. **Prepare** the GSM8K training dataset into the chat-message format required by `GRPOTrainer`.
3. **Design reward functions** that guide the model toward producing well-formatted, correct Chain-of-Thought reasoning.
4. **Train** the model using GRPO with your reward functions. 
5. **Evaluate** the fine-tuned model on the full GSM8K test set.

## Prerequisites

Set Up the Virtual Environment on [EECS JupyterHub (JHub)](https://hub.comp-teach.qmul.ac.uk/)

1. Log in to [EECS JHub](https://hub.comp-teach.qmul.ac.uk/) using your QMUL credentials. Search and start the "EBU6505 Reasoning and Agents" instance.
2. Follow Step 3 to Step 7 in the README.md in the [reasoning-and-agents-education repo](https://github.com/chaoshu1201/reasoning-and-agents-education) to set up and activate the virtual environment on EECS JHub. You can clone either the [reasoning-and-agents-education repo](https://github.com/chaoshu1201/reasoning-and-agents-education) or your forked repo for this project. If you clone your forked repo, please make sure you haven't changed the `pyproject.toml` file and `uv.lock` file, i.e., you haven't installed/upgraded any packages in your forked repo. 
3. Install an interpreter for Jupyter notebooks. Make sure your venv is activated before running the command.

    ```bash
    python -m ipykernel install --user --name reasoning-llms
    ```

4. Upload this notebook to the root of the repo.
5. Before you run the notebook, make sure you choose `reasoning-llms` as the kernel.

> ‼️ Important Note:
> 
>  **The notebook you submit will be tested under the virtual environment specified in the `pyproject.toml` file and `uv.lock` file in the [reasoning-and-agents-education repo](https://github.com/chaoshu1201/reasoning-and-agents-education). Please make sure you strictly follow the instructions to set up a virtual environment specified in the `pyproject.toml` file and `uv.lock` file in the [reasoning-and-agents-education repo](https://github.com/chaoshu1201/reasoning-and-agents-education). If your notebook fails during tests due to inconsistent virtual environment, you will receive 0 mark for the project.**
> 
> Although the same `pyproject.toml` file and `uv.lock` file are used to set up virtual environments on your laptop and JHub, the environments are different. On JHub, where we use Linux and NVIDIA GPUs, `unsloth` and `vllm` will be installed for memory-efficient fine-tuning and high-throughput inference, respectively.

## Project Instructions

In this project:
- **Restricted Environment**: You will work in a restricted environment. Your notebook/script can only use the pre-downloaded `Qwen2-1.5B-Instruct` model and the `GSM8k` dataset.
- **Limited Hardware Resources**: You will work with limited hardware resources. You have a maximum of **50 GB storage*** space and **16 GB GPU VRAM****. This constraint mirrors the real-world limitations of industrial LLM development. You need to know the gap between theory and practice, how the libraries/tools you use are designed in a way that allows you complete your tasks with limited hardware resources.
- **Time Constraints**: Fine-tuning and evaluating LLMs are computationally intensive tasks, even on high-performance GPUs. **Do not delay your project execution until the deadline is imminent**, as training runs and comprehensive evaluations require significant time to complete.
- **Strict Compliance**: You **MUST** follow the instructions in this notebook:
    - You must use certain settings/configurations if instructed in the notebook.
    - You must use certain functions if instructed in the notebook.
    - **No Monkey-Patching or Framework Bypassing**: You **MUST NOT** monkey-patch, wrap, override, or replace any methods or attributes on `model` (such as `model.fast_generate`), `tokenizer`, `vllm_engine`, or any provided evaluation/helper functions. All problem-solving logic during evaluation must be implemented strictly inside `solve_single_problem()`.
    - **Faithful Output Logging**: In `solve_single_problem()`, `text_responses` (`"response"`) must record the raw `.text` from `completions` returned by `llm.fast_generate()` without omission or modification.
    - **No Artificial Inflation of Metrics**: You **MUST NOT** introduce steps whose sole purpose is to artificially inflate evaluation metrics rather than contribute to solving the task.
    - **Training and Evaluation Integrity**: You **MUST** follow sound model-training and evaluation practices, including preventing test-set leakage. For example, you **MUST NOT** use test-set questions or their solutions/answers as training data or as examples in training or evaluation prompts, including paraphrased versions.
    - **Penalty for Non-Compliance**: Violating any of the above compliance rules will result in **0 marks for the relevant code question(s) (Q1.2, Q2.1 and/or Q2.2)**. Evaluation results affected by the violation are invalid and will receive **no marks for the corresponding result-based criteria**. Unaffected questions and criteria will be marked normally.

  While GenAI can easily generate code from scratch, professional environments typically require you to contribute to a complex, existing codebase. You are expected to implement features within the provided framework rather than rewriting the system, a critical skill for working in big tech companies.
- Language Standard: All responses and code comments within the notebook MUST be written in English. Use of other languages will result in a loss of marks.
- You are expected to refer to the [GRPOTrainer](https://huggingface.co/docs/trl/v1.3.0/en/grpo_trainer#grpo-trainer) documents to learn and understand the usage of this library (with the help from AI) during the development of the project, as what you will do in your own project and in your future career. Although you are also encouraged to use GenAI assistants and/or coding agents to help with the development, you should use critical thinking and verify the solutions generated by AI against relevant documents, to ensure you fully understand the solutions provided by AI.

> 💡 *Note:
>
> The initial storage space is 5 GB and will be expanded automatically based on your usage. When your remaining storage is lower than 30% of your current space, your storage will be added by another 5 GB in a couple of minutes.
> 
> Before you start you project, we recoommend you mannually expand your storage to at least 40 GB. You can upload a large files (e.g., 4.5 GB) to your space, wait for 5 minutes, duplicate that file (e.g., use `cp` command: `cp your_large_file your_large_file_1`). Keep duplicate the large file for 7 times, you should get a storage space of 40 GB. After that you can delete all the files you upload and duplicate. The storage size will ramain unchanged.

> 💡 **Note:
>
> Every student should be allocated to a GPU slice with 16 GB VRAM. The GPU types may vary. Sometimes, you will get more than 16 GB VRAM due to a race condition on the server side (Lucky you!). If that happens, it is recommended that you stop your server and/or log out, relaunch the server to get a 16 GB VRAM GPU slice. You can choose to enjoy your luck with more VRAM in the current server instance, but you need to bear in mind that your recipe must work with less than 16 GB VRAM and your recipe will be tested with 16 GB VRAM. Exceeding the 16 GB VRAM limitation will invalidate your recipe and evaluation results during marking.

**Keep your JHub server alive when training and evaluation for a long time**

When you plan to perform a training and evaluation that will take a long time (e.g., > 10 hrs) and you need to leave the JHub page and cannot keep the JHub page open for such a long time, you may need to keep the server alive when you leave the JHub page by using [jupyter-keepalive](https://github.com/minrk/jupyter-keepalive).

1. Deactivate your uv venv. (make sure your uv venv is NOT activated)
2. Run the following command to install `jupyter-keepalive` **in the system environment**.

    ```bash
    pip install jupyter-keepalive
    ```

3. Stop and log out the server, then log back in.
4. Follow the [How to Use](https://github.com/minrk/jupyter-keepalive#how-to-use) instructions of `jupyter-keepalive`.
5. Once a keep-alive session is active, a countdown timer appears in the bottom status bar of JupyterLab showing the remaining time. When no keep-alive session is running, the timer is hidden.

Good Practice
- Only use the keep-alive feature when necessary (e.g. long-running jobs or temporary breaks).
- Remember to stop the timer when you’re finished to free up system resources.

**Running long experiments without keeping your browser open**

With **keepalive active**, computation can continue on the JHub server after you close your browser. However, cell outputs generated while you are disconnected may be missing from the notebook when you return.

**1. During training**

After you execute the training cell and check that training is progressing and `grpo_output/training_stats.csv` is updating. You may then close your browser (but do not click Stop My Server, shut down the kernel) and leave training running if the training takes too long and you cannot keep the JHub tab open.

For submission purposes, the automatically saved training log, `training_stats.csv`, is accepted as equivalent to the training cell output, which will be included in your submission and missing training cell output in the notebook is acceptable.

When you reconnect to the server the JHub should bring you back to the notebook with the same kernel activated, so the trained model remains available. Wait for training and adapter saving to finish before running the evaluation cells.

**2. During evaluation**

Keep the notebook page open if you want to see and retain evaluation outputs, and save the notebook afterwards.

During development, you may also disconnect while evaluation runs and inspect the saved JSON logs when you return. These files are written after all evaluation rounds for each model finish. If you disconnect, some evaluation outputs may be missing from the notebook. The option below lets you capture them for your final submission.

**3. Optional final run for submission**

Use this option when you are ready to prepare your submission but do not yet have a notebook with **all cell outputs captured (except the training cell output)** and **results consistent with the training and evaluation logs you will submit**.

Once your code and recipe are finalised, `nbconvert` can produce this notebook without requiring you to keep your browser open. If your notebook already meets these requirements, an additional run is unnecessary.

**This starts a fresh execution of the whole notebook, including training.** It does not resume your existing run or recover its missing outputs.

To start the final run:

1. Finish your current experiments and save the notebook with all exercise code completed.
2. Shut down the notebook's existing kernel through JupyterLab's **Running** panel. This releases GPU memory for the new kernel that `nbconvert` will start. **Leave the JHub server running.**
3. Set the keepalive timer long enough to cover the full training and evaluation run.
4. Open a **JHub terminal**, go to the directory containing your notebook, and run:

```bash
nohup uv run jupyter nbconvert \
  --to notebook \
  --execute \
  --ExecutePreprocessor.timeout=-1 \
  --ExecutePreprocessor.kernel_name=reasoning-llms \
  P02_Enhancing_Reasoning_Capability_in_LLMs_student.ipynb \
  --output P02_Enhancing_Reasoning_Capability_in_LLMs_student_completed.ipynb \
  > training.log 2>&1 < /dev/null &
```

Check that the run has started successfully before closing your browser. You can monitor training through `training_stats.csv` and inspect `training.log` for execution errors.

After successful completion, the `_student_completed.ipynb` file will contain the executed notebook and its cell outputs. Open it, check the results, and use this completed copy for submission.


## Project Objectives

The objective of this project is to comprehensively apply the reasoning techniques you have learned through L01 to L05 (except L03 & L04) to improve the reasoning capability of the `Qwen2-1.5B-Instruct` model, so that:
1. The improved model can correctly solve as many problems in the GSM8k dataset as possible (**improve the mean problem-solving rate $\bar{R_s}$**).
2. The improved model achieves high per-completion accuracy on its generated outputs (**improve the mean per-completion accuracy $\bar{R_c}$**).
3. The improved model can output reasoning and answers following the exact format below (**improve the mean correct reasoning format rate $\bar{R_f}$**).

   *Expected Output Format:*
   ```text
   <think>[Your step-by-step reasoning process here]</think>
   <answer>[Final numerical answer here]</answer>
   ```

   *Example:*
   ```text
   "<think> Janet's ducks lay 16 eggs per day. She eats three for breakfast, so she has 16 - 3 = 13 eggs left. She bakes muffins with 4 eggs, so she has 13 - 4 = 9 eggs left. She sells the remaining eggs at $2 per egg, so she makes 9 * $2 = $18 every day at the farmers' market. </think>\n<answer>18</answer>"
   ```

## Marking Rubric

Your **project mark $S$ (UK Scale)** will be calculated as:

$$ S = B + \bar{R_f} \times 10 + \min\Big(\max\big((\bar{R_s} - 0.60), 0\big) \times \frac{1000}{3},\ 50\Big) + \min\Big(\max\big((\bar{R_c} - 0.55), 0\big) \times 100,\ 10\Big) $$

Where:

$B \in [0, 30]$ is the mark for your notebook (code) report. Most of them are not performance-relevant. Marks will be awarded as long as the code and answers are correct and reasonable.

$\bar{R_f} \in [0, 1]$ is the mean correct format rate in your evaluation results.

$\bar{R_s} \in [0, 1]$ is the mean problem-solving rate in your evaluation results.

$\bar{R_c} \in [0, 1]$ is the mean per-completion accuracy in your evaluation results.

How $\bar{R_s}$, $\bar{R_c}$, and $\bar{R_f}$ are calculated: Because generative LLMs exhibit variance in their outputs (especially when sampling with temperature), your model is evaluated over $N$ independent rounds (In this project, we use $N=4$) across the entire GSM8k test set to ensure a robust and fair assessment.

**Mean Problem-Solving Rate ($\bar{R_s}$)**: For each evaluation round $i$, a solve rate $R_{s,i}$ is calculated as the proportion of test problems where the final predicted numerical answer (`predicted`) correctly matches the ground truth (`expected`). The overall mean is the average across all $N$ rounds: $$ \bar{R_s} = \frac{1}{N} \sum_{i=1}^{N} R_{s,i} $$

**Mean Per-Completion Accuracy ($\bar{R_c}$)**: For each evaluation round $i$, a completion accuracy $R_{c,i}$ is calculated as the average, across all test problems, of the proportion of generated model completion(s) in `text_responses` (`response`) whose extracted answer (via `extract_answer()`) correctly matches the ground truth (`expected`). The contribution of a problem is defined as zero when the attempt fails or produces no completion. The overall mean is the average across all $N$ rounds: $$ \bar{R_c} = \frac{1}{N} \sum_{i=1}^{N} R_{c,i} $$

**Mean Correct Format Rate ($\bar{R_f}$)**: For each evaluation round $i$, a format rate $R_{f,i}$ is calculated as the proportion of test problems whose model responses successfully comply with the required structural tags (i.e., containing both the `<think>...</think>` reasoning block and the `<answer>...</answer>` solution block). The overall mean is the average across all $N$ rounds: $$ \bar{R_f} = \frac{1}{N} \sum_{i=1}^{N} R_{f,i} $$

> **Note:** These performance targets are challenging but achievable in the provided JHub environment using the framework supplied in the module repository. High marks, including full marks, require a thorough understanding of deep learning and the LLM reasoning techniques introduced in this module, strong analytical and critical thinking skills, and a well-developed solution.


# PART I: GRPO Fine-Tuning
---

## Imports and Seed Setup

You may add more imports if necessary (but not to install new libraries). You MUST add all imports in the following cell.

In [1]:
import os
import re
import json
import csv
import math
import time
import random
from datetime import datetime

import numpy as np
from tqdm import tqdm

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from vllm import SamplingParams
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback

[HAMI-core Msg(4589:140236260609344:libvgpu.c:839)]: Initializing.....
[HAMI-core Warn(4589:140236260609344:multiprocess_memory_limit.c:548)]: Kick dead proc 2530
[HAMI-core Warn(4589:140236260609344:multiprocess_memory_limit.c:548)]: Kick dead proc 2884


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[HAMI-core Msg(4589:140236260609344:libvgpu.c:855)]: Initialized


🦥 Unsloth Zoo will now patch everything to make training faster!


## Student ID and Recipe Configuration

Set your Student ID and configure the experiment recipe dictionary `recipe_dic`. The `recipe_dic` packs all experiment hyper-parameters into a single dictionary so that every setting is captured in one place.

> 💡 Note:
> 
> - You **MUST** set your `STUDENT_ID` below.
> - Do **NOT** change the `seed` value – a fixed seed ensures fair comparison across students.

⛔ DO NOT change the next cell

In [2]:
# Set random seeds for reproducibility
seed = 3407
SEED_STRIDE = 1000

# Model path
model_name = os.path.expandvars("$HOME/reasoning-and-agents-shared/huggingface/models/Qwen2-1.5B-Instruct")

# Output directory for fine-tuned lora adapter and evaluation results
output_dir = "./grpo_output"
os.makedirs(output_dir, exist_ok=True)

# Set global random seeds for reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
output_dir = "./grpo_output_formal_run01"
os.makedirs(output_dir, exist_ok=True)

Please set your QMUL student ID

In [4]:
# TODO: !!IMPORTANT!! Set your QMUL Student ID
STUDENT_ID = "241119954"  # Replace with your QMUL student ID

Please set your experiment parameters. For the provided values, you can change them when you are testing your project, but you MUST change them back to the original values in your final run for submission. 

> 💡 Note:
>
> You don't need to build a complete recipe at the beginning. You can add more hyperparameters and tweak their values during the optimisation of your model.

In [5]:

# Experiment recipe – formal training configuration
recipe_dic = {
    "identity_config": {
        "student_id": STUDENT_ID,
    },

    "model_config": {
        "model": model_name,
        "lora_dir": None,
        "max_seq_length": 2048,
        "dtype": None,
        "load_in_4bit": True,
        "fast_inference": True,
        "gpu_memory_utilization": 0.30,
        "max_lora_rank": 16,
        "random_state": seed,
        "use_gradient_checkpointing": "unsloth",
        "lora_r": 16,
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        "lora_alpha": 16,
        "lora_dropout": 0.0,
        "bias": "none",
        "use_rslora": False,
    },

    "dataset_config": {},

    "grpo_config": {
        # Basic configuration
        "seed": seed,
        "report_to": "none",
        "output_dir": os.path.join(output_dir, "checkpoints"),

        # Training duration
        "num_train_epochs": 1,
        "max_steps": -1,

        # Batch configuration
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 1,
        "num_generations": 4,

        # Learning configuration
        "learning_rate": 5e-6,
        "adam_beta1": 0.9,
        "adam_beta2": 0.99,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_8bit",
        "max_grad_norm": 0.1,

        # Generation configuration
        "max_prompt_length": 512,
        "max_completion_length": 384,
        "temperature": 0.8,
        "top_p": 0.95,

        # GRPO configuration
        "beta": 0.04,
        "remove_unused_columns": False,

        # Precision and memory
        "bf16": True,
        "fp16": False,
        "gradient_checkpointing": True,

        # Logging and checkpointing
        "logging_strategy": "steps",
        "logging_steps": 10,
        "save_strategy": "steps",
        "save_steps": 100,
        "save_total_limit": 1,

        # vLLM
        "use_vllm": True,
        "vllm_mode": "colocate",
    },

    "eval_config": {
        "num_rounds": 4,
        "seed": seed,
        "eval_model": "both",
        "eval_target": None,
    },

    "sampling_params": {},
}


## Load the Model

In this section, you will load a pre-trained instruct model Qwen2-1.5B-Instruct using Unsloth's `FastLanguageModel` and attach a LoRA (Low-Rank Adaptation) adapter for parameter-efficient fine-tuning.

In this project, we use [Unsloth](https://unsloth.ai/) because it can make fine-tuning LLMs faster and more memory-efficient, which is especially useful when VRAM is limited.

**[Q1.1]** Load the model and attach the LoRA adapter using the parameters from `recipe_dic`. **[4 Marks]**

In [6]:

print("[1/4] Loading model...")

# Get model configurations from recipe_dic
model_cfg = recipe_dic["model_config"]

# Load the pre-trained model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_cfg["model"],
    max_seq_length=model_cfg["max_seq_length"],
    dtype=model_cfg["dtype"],
    load_in_4bit=model_cfg["load_in_4bit"],
    fast_inference=model_cfg["fast_inference"],
    gpu_memory_utilization=model_cfg["gpu_memory_utilization"],
    max_lora_rank=model_cfg["max_lora_rank"],
    random_state=model_cfg["random_state"],
    use_gradient_checkpointing=model_cfg["use_gradient_checkpointing"],
)

# Attach LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=model_cfg["lora_r"],
    target_modules=model_cfg["target_modules"],
    lora_alpha=model_cfg["lora_alpha"],
    lora_dropout=model_cfg["lora_dropout"],
    bias=model_cfg["bias"],
    use_gradient_checkpointing=model_cfg["use_gradient_checkpointing"],
    random_state=model_cfg["random_state"],
    max_seq_length=model_cfg["max_seq_length"],
    use_rslora=model_cfg["use_rslora"],
)

print("  Model loaded.\n")


[1/4] Loading model...
INFO 11-25 14:49:30 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.10: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 16.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: FlashInfer requires JIT compilation but nvcc (CUDA compiler) and ninja (build tool) are not found.
  vLLM will use FLASH_ATTN attention + PyTorch sampler instead (works fine).
  To enable FlashInfer, install the missing tools:
    nvcc  - install the CUDA toolkit or set CUDA_HOME to your CUDA installation
    ninja - pip install ninja
  To silence this warning: set UNSLOTH_VLLM_NO_FLASHINFER=1
Unsloth: vLLM l

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 11-25 14:49:36 [punica_selector.py:20] Using PunicaWrapperGPU.
INFO 11-25 14:49:37 [gpu_model_runner.py:4566] Model loading took 1.11 GiB memory and 4.099595 seconds
INFO 11-25 14:49:42 [backends.py:988] Using cache directory: /home/jovyan/.cache/vllm/torch_compile_cache/226794bf4c/rank_0_0/backbone for vLLM's torch.compile
INFO 11-25 14:49:42 [backends.py:1048] Dynamo bytecode transform time: 4.05 s
INFO 11-25 14:49:46 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 2048) from the cache, took 2.104 s
INFO 11-25 14:49:46 [monitor.py:48] torch.compile took 7.25 s in total
INFO 11-25 14:49:46 [decorators.py:296] Directly load AOT compilation from path /home/jovyan/.cache/vllm/torch_compile_cache/torch_aot_compile/e3a821474929717b83360781e0a3a0bcc95e09e68046a10e8611870474040804/rank_0_0/model
INFO 11-25 14:49:46 [monitor.py:76] Initial profiling/warmup run took 0.23 s
INFO 11-25 14:49:46 [kv_cache_utils.py:826] Overriding num_gpu_blocks=0 with num_gpu_bloc

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 14/14 [00:03<00:00,  3.78it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 10/10 [00:02<00:00,  4.30it/s]

INFO 11-25 14:49:57 [gpu_model_runner.py:5746] Graph capturing finished in 6 secs, took 0.06 GiB
INFO 11-25 14:49:57 [vllm_utils.py:736] Unsloth: Patched vLLM v1 graph capture finished in 6 secs.


INFO 11-25 14:49:58 [gpu_worker.py:617] CUDA graph pool memory: 0.06 GiB (actual), 0.08 GiB (estimated), difference: 0.02 GiB (30.3%).
INFO 11-25 14:49:58 [core.py:281] init engine (profile, create kv cache, warmup model) took 20.92 seconds
INFO 11-25 14:49:59 [llm.py:391] Supported tasks: ('generate',)


Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at /home/jovyan/reasoning-and-agents-shared/huggingface/models/Qwen2-1.5B-Instruct and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Unsloth: Just some info: will skip parsing ['k_norm', 'post_feedforward_layernorm', 'post_attention_layernorm', 'norm1', 'norm2', 'norm', 'q_norm', 'attention_norm', 'post_layernorm', 'layer_norm2', 'ffn_norm', 'layer_norm1', 'pre_feedforward_layernorm', 'input_layernorm']
Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['k_norm', 'cross_attn_post_attention_layernorm', 'post_feedforward_layernorm', 'post_attention_layernorm', 'norm1', 'norm2', 'norm', 'q_norm', 'attention_norm', 'post_layernorm', 'layer_norm2', 'ffn_norm', 'layer_norm1', 'pre_feedforward_layernorm', 'cross_attn_input_layernorm', 'input_layernorm']


Unsloth 2026.3.10 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


  Model loaded.



Data Preparation
---

In this section, you will:

1. Define **reasoning and solution tags** (`<think>`, `</think>`, `<answer>`, `</answer>`) that structure the model's output format.
2. Create a **system prompt** instructing the model to reason step-by-step and wrap its output in the defined tags.
3. Load the **[GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)** and preprocess the training split into the chat-message format expected by `GRPOTrainer`:
   - Each example should have a `"prompt"` field (list of system + user messages) and an `"answer"` field (the ground-truth numerical answer extracted from `"#### X"`).

Firstly, let's define **reasoning and solution tags** (`<think>`, `</think>`, `<answer>`, `</answer>`) that structure the model's output format.

⛔ DO NOT change the next cell

In [7]:
# Define reasoning_start, reasoning_end, solution_start, solution_end tags
reasoning_start = "<think>"
reasoning_end = "</think>"
solution_start = "<answer>"
solution_end = "</answer>"

**[Q1.2]** Create the system prompt by using the defined the reasoning/solution tags. **[2 Marks]**

In [8]:
# Create the system prompt using the defined tags

system_prompt = f"""
You are a helpful mathematical reasoning assistant.

Your task is to solve grade-school mathematical word problems.

Follow these instructions:
1. Read the question carefully.
2. Break the problem into logical calculation steps.
3. Show clear and concise reasoning.
4. Check your calculations before giving the final answer.
5. The final answer must be a numerical value.

You MUST follow this exact output format:

{reasoning_start}
Write your step-by-step mathematical reasoning here.
{reasoning_end}
{solution_start}
Write the final numerical answer here.
{solution_end}

Do not include any additional text outside these tags.
"""

print("System prompt created successfully.")
print(system_prompt)

System prompt created successfully.

You are a helpful mathematical reasoning assistant.

Your task is to solve grade-school mathematical word problems.

Follow these instructions:
1. Read the question carefully.
2. Break the problem into logical calculation steps.
3. Show clear and concise reasoning.
4. Check your calculations before giving the final answer.
5. The final answer must be a numerical value.

You MUST follow this exact output format:

<think>
Write your step-by-step mathematical reasoning here.
</think>
<answer>
Write the final numerical answer here.
</answer>

Do not include any additional text outside these tags.



Before we load the GSM8k dataset, let's define a helper function to extract the answer after "####". You must use the `extract_hash_answer()` helper function when necessary.

⛔ DO NOT change the next cell

In [9]:
def extract_hash_answer(text):
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

Next, we load the GSM8K dataset, preprocess the training split into chat-message format. It is implemented for you.

In [10]:
print("[2/4] Preparing dataset...")

gsm8k = load_dataset("openai/gsm8k", "main")

train_dataset = gsm8k["train"].map(lambda x: {
    "prompt": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": x["question"]},
    ],
    "answer": extract_hash_answer(x["answer"]),
})

recipe_dic["dataset_config"]["train_size"] = len(train_dataset)

print("  Dataset prepared.\n")

Using the latest cached version of the dataset since openai/gsm8k couldn't be found on the Hugging Face Hub (offline mode is enabled).
[datasets.load|WARNING]Using the latest cached version of the dataset since openai/gsm8k couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'main' at /home/jovyan/.cache/huggingface/datasets/openai___gsm8k/main/0.0.0/cc7b047b6e5bb11b4f1af84efc572db110a51b3c (last modified on Wed Nov 25 09:19:47 2026).
[datasets.packaged_modules.cache.cache|WARNING]Found the latest cached dataset configuration 'main' at /home/jovyan/.cache/huggingface/datasets/openai___gsm8k/main/0.0.0/cc7b047b6e5bb11b4f1af84efc572db110a51b3c (last modified on Wed Nov 25 09:19:47 2026).


[2/4] Preparing dataset...
  Dataset prepared.



## Reward Functions

In this section, you will design **reward functions** that the `GRPOTrainer` uses to score each generated completion. These rewards shape the model's learning signal.

You should define **regex patterns** to match the expected output format and extract answers from the outputs, then implement reward functions that encourage both correct formatting and correct answers, aiming to achieve the objectives defined in the "**Project Objectives**" section. A large proportion of your project marks is determined by how well the objectives are achieved, as defined in the "**Marking Rubric**" section. 

Each reward function receives `completions` (a list of generated responses) and optionally `prompts` and `answer` (ground truth), and returns a list of numerical scores.

> 💡 Note:
> 
> You are free to design your own reward functions and scoring schemes. The number of reward functions, their names, and the reward values are all up to you. This is a key part of your recipe design.

A Reference definition of regex patterns for format matching and answer extraction are provided in the next cell for you to use in this project. You are free to define your own regex pattern, but your fine-tuned model and recipe will be evaluated using the following regrex patterns.

⛔ DO NOT change the next cell

In [15]:
# Define regex pattern(s) to match the full <think>...</think><answer>...</answer> format
match_format = re.compile(
    rf"^[\s]{{0,}}"
    rf"{re.escape(reasoning_start)}.+?{re.escape(reasoning_end)}.*?"
    rf"{re.escape(solution_start)}(.+?){re.escape(solution_end)}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL
)

# Define regex pattern(s) to extract numerical answers from <answer> tags
match_numbers = re.compile(
    re.escape(solution_start) + r".*?([\d\.\,]{1,})",
    flags=re.MULTILINE | re.DOTALL
)

**[Q1.3]** Implement your reward functions. You should implement **at least two** reward functions: one for format checking and one for answer correctness. You may add more. Each reward function must follow the signature expected by [`GRPOTrainer`](https://huggingface.co/docs/trl/v1.3.0/en/grpo_trainer#using-a-custom-reward-function). **[8 Marks]**

In [12]:

print("[3/4] Setting up reward functions...")


# Helper function to extract text from conversational completions
def get_completion_text(completion):
    if isinstance(completion, str):
        return completion

    if isinstance(completion, dict):
        return str(completion.get("content", ""))

    if isinstance(completion, list):
        for message in reversed(completion):
            if isinstance(message, dict) and "content" in message:
                return str(message["content"])

    return str(completion)


# Reward 1: Check the required reasoning and answer format
def format_reward_func(completions, **kwargs):
    rewards = []

    for completion in completions:
        text = get_completion_text(completion)

        if match_format.search(text):
            rewards.append(1.0)
        else:
            rewards.append(0.0)

    return rewards


# Reward 2: Check numerical answer correctness
def answer_correctness_reward_func(completions, answer, **kwargs):
    if isinstance(answer, (str, int, float)):
        answer = [answer]

    if answer is None or len(completions) != len(answer):
        raise ValueError("Completions and reference answers must have the same length.")

    rewards = []

    for completion, expected in zip(completions, answer):
        text = get_completion_text(completion)

        predicted_match = match_numbers.search(text)

        if predicted_match is None or expected is None:
            rewards.append(0.0)
            continue

        predicted = predicted_match.group(1).replace(",", "").strip()
        expected = str(expected).replace(",", "").strip()

        try:
            is_correct = math.isclose(
                float(predicted),
                float(expected),
                rel_tol=0.0,
                abs_tol=1e-8
            )
        except (ValueError, TypeError):
            is_correct = predicted == expected

        rewards.append(1.0 if is_correct else 0.0)

    return rewards


print("  Reward functions defined.\n")


[3/4] Setting up reward functions...
  Reward functions defined.



## GRPO Training

In this section, you will:

1. **Configure `GRPOConfig`** with the training hyper-parameters from `recipe_dic`.
2. **Create the `GRPOTrainer`** with your model, tokenizer, reward functions, training args, dataset, and the CSV callback.
3. **Run training** and time the process.
4. **Save the LoRA adapter** for later evaluation and submission.

Before configuring the trainer, let's implement the `CSVLoggingCallback` class. The callback will log general training metrics. (To make it easier and consistent in marking across all students, we don't use Weights & Biases in this project.)

⛔ DO NOT change the next cell

In [13]:
# CSV logging callback
training_log_path = os.path.join(output_dir, "training_stats.csv")

class CSVLoggingCallback(TrainerCallback):
    def __init__(self, path):
        self.path = path
        self._file = None
        self._writer = None

    def on_train_begin(self, args, state, control, **kwargs):
        self._file = open(self.path, "w", newline="")
        self._writer = csv.DictWriter(self._file, fieldnames=[
            "step", "epoch", "loss", "learning_rate", "grad_norm",
            "reward", "reward_std",
            "completion_length", "kl",
        ])
        self._writer.writeheader()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and self._writer:
            row = {
                "step": state.global_step,
                "epoch": round(state.epoch, 4) if state.epoch else "",
                "loss": logs.get("loss", ""),
                "learning_rate": logs.get("learning_rate", ""),
                "grad_norm": logs.get("grad_norm", ""),
                "reward": logs.get("reward", ""),
                "reward_std": logs.get("reward_std", ""),
                "completion_length": logs.get("completion_length", ""),
                "kl": logs.get("kl", ""),
            }
            self._writer.writerow(row)
            self._file.flush()

    def on_train_end(self, args, state, control, **kwargs):
        if self._file:
            self._file.close()


**[Q1.4]** Configure `GRPOConfig`, create `GRPOTrainer`, run training, and save the LoRA adapter. **[5 Marks]**

> 💡 Note:
>
> - Use `recipe_dic` values for all hyper-parameters.
> - Pass your list of reward functions to `GRPOTrainer`.
> - Save the LoRA adapter to `{output_dir}/grpo_saved_lora/` using `model.save_lora()`.

In [14]:
print("[4/4] Starting GRPO training...")

recipe_dic["dataset_config"]["train_size"] = len(train_dataset)

grpo_kwargs = {
    key: value
    for key, value in recipe_dic["grpo_config"].items()
    if value is not None
}

assert training_log_path == os.path.join(output_dir, "training_stats.csv"), (
    "Run the CSVLoggingCallback cell after setting the output directory."
)

training_args = GRPOConfig(**grpo_kwargs)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        answer_correctness_reward_func,
    ],
    args=training_args,
    train_dataset=train_dataset,
    callbacks=[CSVLoggingCallback(training_log_path)],
)

start_time = time.time()
trainer.train()

elapsed = time.time() - start_time
print(f"  GRPO training done. Time: {elapsed:.1f}s")
print(f"  Training stats saved to: {training_log_path}")

lora_dir = os.path.join(output_dir, "grpo_saved_lora")
model.save_lora(lora_dir)
print(f"  LoRA adapter saved to: {lora_dir}")

[4/4] Starting GRPO training...


[HAMI-core Msg(3037:140642741073728:multiprocess_memory_limit.c:455)]: Calling exit handler 3037
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 7,473
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


WARNING 11-25 09:50:59 [input_processor.py:141] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / answer_correctness_reward_func / mean,rewards / answer_correctness_reward_func / std
10,0.036800,0.325000,0.411213,241.000000,183.000000,313.300000,0.050000,234.425000,183.000000,302.100000,0.000199,0.250000,0.357735,0.075000,0.107735
20,0.085400,0.300000,0.315470,206.225000,138.200000,281.700000,0.025000,200.741667,138.200000,266.800000,0.000528,0.275000,0.307735,0.025000,0.050000
30,0.044700,0.375000,0.641485,163.625000,97.600000,247.800000,0.000000,163.625000,97.600000,247.800000,0.000589,0.225000,0.365470,0.150000,0.300000
40,-0.002800,0.350000,0.361213,181.225000,136.500000,229.300000,0.000000,181.225000,136.500000,229.300000,0.000468,0.250000,0.215470,0.100000,0.157735
50,-0.007500,0.400000,0.415470,190.375000,138.600000,250.900000,0.050000,183.483334,138.600000,234.100000,0.000502,0.300000,0.215470,0.100000,0.200000
60,0.005100,0.550000,0.534418,172.400000,126.100000,228.900000,0.025000,169.291669,126.100000,223.300000,0.000594,0.450000,0.430940,0.100000,0.115470
70,0.047900,0.375000,0.480940,223.575000,151.600000,286.200000,0.050000,219.466669,151.600000,280.200000,0.000555,0.250000,0.373205,0.125000,0.250000
80,0.078800,0.525000,0.544239,200.650000,134.800000,281.800000,0.100000,184.441667,134.800000,243.700000,0.000606,0.400000,0.430940,0.125000,0.207735
90,0.033300,0.175000,0.295743,154.700000,105.000000,212.300000,0.025000,150.883334,105.000000,207.500000,0.000615,0.125000,0.207735,0.050000,0.100000
100,0.060300,0.425000,0.514691,166.225000,107.400000,229.000000,0.025000,162.191667,107.400000,216.200000,0.001188,0.300000,0.373205,0.125000,0.207735


KeyboardInterrupt: 

# Part II: Evaluation
---

In this section, you will evaluate the GRPO-fine-tuned model on the **full GSM8K test set** over multiple rounds. This involves:

1. **Complete the implementation**: Most of the evaluation functions are provided, except for a few parts of `solve_single_problem()` — a function that generates completions for a single problem using unsloth's `fast_generate`. You are required to complete the implementation of the `solve_single_problem()` function.
2. **Running the evaluation** with the relevant parameters from `recipe_dic`. The provided evaluation functions are the same functions that will be used in marking. By running evaluations using these functions, you can have a good estimation of the marks you can get based on your evaluation results. 
3. **Saving results** — Results and logs required for submission will be automatically saved after you run the evaluations.

Let's first define the function to extract the answer from the model output. You MUST use this function in `solve_single_problem()` to extract the answer from the outputs of the model.

⛔ DO NOT change the next cell

In [11]:
# Define the function to extract the answer from the model output
def extract_answer(output):
    fmt_match = match_format.search(output)
    if fmt_match:
        return fmt_match.group(1).strip(), True   # (answer, has_format)
    num_match = match_numbers.search(output)
    if num_match:
        return num_match.group(1).strip(), False
    return None, False

**[Q2.1]** Complete the `solve_single_problem()` function following the TODO instructions. **[5 Marks]**

> ⚠️ **Note:**
> - All problem-solving logic during evaluation must be implemented inside `solve_single_problem()` using `llm.fast_generate()`. Do **not** wrap, override, replace, or monkey-patch `model.fast_generate` or `llm.fast_generate` anywhere in the notebook, including inside this function.
> - `text_responses` (`"response"`) must record the raw `.text` from `completions` without omission or modification. You must not introduce steps whose sole purpose is to artificially inflate evaluation metrics rather than contribute to solving the task. Violating any of the above compliance rules will result in **0 marks for Q2.1**. Evaluation results affected by the violation are invalid and will receive **no marks for the corresponding result-based criteria**.

In [12]:
def solve_single_problem(
    problem, prompt_template, llm, sampling_params, lora_request=None, *, request_seed
):
    """Solve a single problem in the dataset.
    Arguments:
        problem (dict): The problem to solve.
        prompt_template (str): The prompt to be used as the system message.
        llm: The language model to use.
        sampling_params (dict): Parameters for sampling.
        lora_request (LoRARequest, optional): The LoRA request for fine-tuned evaluation.
            lora_request=None: Inference using the base model; pass a loaded LoRA request for fine-tuned evaluation.
        request_seed (int): Reproducible sampling seed for this problem and round.
    """

    question = problem["question"]
    expected = extract_hash_answer(problem["answer"])

    if expected is None:
        raise ValueError("Missing ground-truth answer")

    messages = [
        {"role": "system", "content": prompt_template},
        {"role": "user", "content": question},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    try:
        sp = SamplingParams(**{**sampling_params, "seed": request_seed})

        completions = llm.fast_generate(
            text,
            sampling_params=sp,
            lora_request=lora_request,
            use_tqdm=False,
        )[0].outputs

        text_responses = [completion.text for completion in completions]

        predicted, has_format = (
            extract_answer(text_responses[0])
            if text_responses
            else (None, False)
        )

        # --- Correctness check ---
        is_correct = False
        if predicted is not None and expected is not None:
            try:
                is_correct = float(predicted.replace(",", "")) == float(expected.replace(",", ""))
            except (ValueError, TypeError):
                is_correct = predicted.strip() == expected.strip()

        return {
            "question": question,
            "expected": expected,
            "predicted": predicted,
            "correct": is_correct,
            "has_format": has_format,
            "response": text_responses,
            "request_seed": request_seed,
        }
    except Exception as error:
        print(f"Error processing example (seed={request_seed}): {error}")
        return {
            "question": question,
            "expected": expected,
            "response": None,
            "request_seed": request_seed,
            "error_type": type(error).__name__,
            "error_message": str(error),
        }

Define other evaluation functions.

⛔ DO NOT change the next cell

In [13]:
def run_evaluation(llm, problems, prompt_template, rounds, sampling_params, lora_request=None, student_id=None):
    """Log failed attempts and score each round against the full test-set size.

    lora_request=None → base model; pass a loaded LoRA request for fine-tuned evaluation.
    """
    all_metrics = []
    all_responses = []
    num_problems = len(problems)
    think_re = re.compile(
        re.escape(reasoning_start) + r"(.*?)" + re.escape(reasoning_end), re.DOTALL
    )

    print(f"\nRunning {rounds} rounds of evaluation...")

    for round_num in range(1, rounds + 1):
        print(f"\n--- Evaluation Round {round_num}/{rounds} ---")
        t0 = time.time()

        results = []
        correct = 0
        format_ok = 0
        total_completion_acc = 0.0
        total_thinking_tokens = 0
        num_errors = 0

        for i, example in enumerate(tqdm(problems, desc=f"Round {round_num}")):
            t_prob = time.time()
            # Derive a distinct reproducible seed for every problem and round.
            request_index = (round_num - 1) * num_problems + i
            request_seed = seed + request_index * SEED_STRIDE
            res = solve_single_problem(
                example, prompt_template, llm, sampling_params,
                lora_request=lora_request, request_seed=request_seed,
            )
            
            res["index"] = i
            res["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            res["elapsed_seconds"] = round(time.time() - t_prob, 2)
            results.append(res)
            if "error_message" in res:
                num_errors += 1
                continue  # The failed attempt remains saved and in the denominator.
            
            if res["correct"]:
                correct += 1
            if res["has_format"]:
                format_ok += 1
            raw_resps = res.get("response")
            if isinstance(raw_resps, str):
                raw_resps = [raw_resps]
            elif not isinstance(raw_resps, list):
                raw_resps = list(raw_resps) if raw_resps is not None else []
            think_toks = 0
            if raw_resps:
                exp_str = str(res["expected"]) if res.get("expected") is not None else None
                comp_hits = 0
                for r_text in raw_resps:
                    if not isinstance(r_text, str):
                        continue
                    if (m := think_re.search(r_text)):
                        think_toks += len(tokenizer.encode(m.group(1), add_special_tokens=False))
                    if exp_str is not None:
                        cand, _ = extract_answer(r_text)
                        if cand is not None:
                            try:
                                is_hit = float(cand.replace(",", "")) == float(exp_str.replace(",", ""))
                            except (ValueError, TypeError):
                                is_hit = cand.strip() == exp_str.strip()
                            if is_hit:
                                comp_hits += 1
                if exp_str is not None:
                    total_completion_acc += comp_hits / len(raw_resps)
                res["n_thinking_tokens"] = round(think_toks / len(raw_resps), 1)
            else:
                res["n_thinking_tokens"] = 0.0
            total_thinking_tokens += res["n_thinking_tokens"]

        elapsed = time.time() - t0
        n = num_problems
        num_generated = n - num_errors
        round_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        round_summary = {
            "round": round_num,
            "student_id": student_id,
            "timestamp": round_timestamp,
            "eval_size": n,
            "correct": correct,
            "num_errors": num_errors,
            "solve_rate": round(correct / n, 4) if n else 0,
            "completion_accuracy": round(total_completion_acc / n, 4) if n else 0,
            "format_rate": round(format_ok / n, 4) if n else 0,
            "avg_thinking_tokens": (
                round(total_thinking_tokens / num_generated, 1) if num_generated else None
            ),
            "elapsed_seconds": round(elapsed, 1)
        }
        all_metrics.append(round_summary)
        all_responses.append(results)
        
        print(f"  Round {round_num}: solve_rate={round_summary['solve_rate']:.4f} "
              f"({round_summary['correct']}/{round_summary['eval_size']}), "
              f"completion_acc={round_summary['completion_accuracy']:.4f}, "
              f"format_rate={round_summary['format_rate']:.4f}, "
              f"avg_thinking_tokens={round_summary['avg_thinking_tokens']}, "
              f"errors={num_errors}, "
              f"time={elapsed:.1f}s")

    solve_rates = [r["solve_rate"] for r in all_metrics]
    completion_accs = [r["completion_accuracy"] for r in all_metrics]
    format_rates = [r["format_rate"] for r in all_metrics]
    thinking_tokens = [r["avg_thinking_tokens"] for r in all_metrics]
    valid_thinking_tokens = [value for value in thinking_tokens if value is not None]

    aggregate_summary = {
        "num_errors": sum(r["num_errors"] for r in all_metrics),
        "solve_rate": {
            "mean": round(float(np.mean(solve_rates)), 4),
            "std_dev": round(float(np.std(solve_rates)), 4),
            "min": round(float(np.min(solve_rates)), 4),
            "max": round(float(np.max(solve_rates)), 4),
            "per_round": solve_rates,
        },
        "completion_accuracy": {
            "mean": round(float(np.mean(completion_accs)), 4),
            "std_dev": round(float(np.std(completion_accs)), 4),
            "min": round(float(np.min(completion_accs)), 4),
            "max": round(float(np.max(completion_accs)), 4),
            "per_round": completion_accs,
        },
        "format_rate": {
            "mean": round(float(np.mean(format_rates)), 4),
            "std_dev": round(float(np.std(format_rates)), 4),
            "per_round": format_rates,
        },
        "avg_thinking_tokens": {
            "mean": (
                round(float(np.mean(valid_thinking_tokens)), 1) if valid_thinking_tokens else None
            ),
            "std_dev": (
                round(float(np.std(valid_thinking_tokens)), 1) if valid_thinking_tokens else None
            ),
            "per_round": thinking_tokens,
        },
    }
    
    return aggregate_summary, all_metrics, all_responses

def run_eval_for_targets(
    eval_model,
    output_dir,
    model,
    lora_dir,
    test_dataset,
    system_prompt,
    recipe_dic,
):
    """Run evaluation for one or more model variants (base / lora / both).

    Args:
        eval_model:          "base", "lora", or "both".
        output_dir:          Root directory for all outputs.
        model:               Loaded FastLanguageModel (vLLM engine).
        lora_dir:            Path to the saved LoRA adapter directory.
        test_dataset:        HuggingFace dataset split to evaluate on (e.g. gsm8k["test"]).
        system_prompt:       system prompt string.
        recipe_dic:          Full config dict (identity_config, model_config,
                             dataset_config, grpo_config, eval_config,
                             sampling_params). Used directly as the "config"
                             block in the saved metrics JSON.
    """
    print(f"  Test set size: {len(test_dataset)}")
    recipe_dic["dataset_config"]["eval_size"] = len(test_dataset)
    recipe_dic["model_config"]["lora_dir"] = lora_dir

    num_eval_rounds = recipe_dic["eval_config"]["num_rounds"]
    sampling_params = recipe_dic["sampling_params"]
    student_id      = recipe_dic["identity_config"]["student_id"]
    model_name      = recipe_dic["model_config"]["model"]

    # ── Iterate over eval targets ─────────────────────────────────────────────
    eval_targets = ["base", "lora"] if eval_model == "both" else [eval_model]

    for eval_target in eval_targets:
        responses_dir = os.path.join(output_dir, f"responses_{eval_target}")
        os.makedirs(responses_dir, exist_ok=True)

        if eval_target == "base":
            lora_request = None
            print("  Evaluating base model (lora_request=None).")
        else:
            if not os.path.isdir(lora_dir):
                raise FileNotFoundError(
                    f"LoRA directory not found: {lora_dir}. "
                    "Run training first or point output_dir to an existing LoRA run."
                )
            lora_request = model.load_lora(lora_dir)
            print(f"  Evaluating LoRA model from: {lora_dir}")

        aggregate_summary, all_rounds, all_responses = run_evaluation(
            llm=model,
            problems=test_dataset,
            prompt_template=system_prompt,
            rounds=num_eval_rounds,
            sampling_params=sampling_params,
            lora_request=lora_request,
            student_id=student_id,
        )

        for round_num, (round_summary, round_results) in enumerate(zip(all_rounds, all_responses), 1):
            round_filename = f"round_{round_num}_responses.json"
            with open(os.path.join(responses_dir, round_filename), "w") as f:
                json.dump(
                    {
                        "student_id": student_id,
                        "timestamp": round_summary["timestamp"],
                        "summary": round_summary,
                        "results": round_results,
                    },
                    f,
                    indent=2,
                    default=str,
                )

        # ── Consolidated metrics file ─────────────────────────────────────────
        recipe_dic["eval_config"]["eval_target"] = eval_target
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_identifier = model_name.replace("/", "-").replace(":", "_")
        metrics_filename = f"metrics_{model_identifier}_{eval_target}_{student_id}_{timestamp}.json"
        metrics_path = os.path.join(output_dir, metrics_filename)

        consolidated = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "config": recipe_dic,
            "results": {
                "summary": aggregate_summary,
                "rounds":  all_rounds,
            }
        }

        with open(metrics_path, "w") as f:
            json.dump(consolidated, f, indent=2, default=str)

        # ── Print final summary ───────────────────────────────────────────────
        print(f"\n{'='*60}")
        print(f"MULTI-ROUND EVALUATION SUMMARY ({num_eval_rounds} rounds)")
        print(f"{'='*60}")
        print(f"  Student ID:          {student_id}")
        print(f"  Model:               {model_name}")
        print(f"  Eval model:          {eval_target}")
        print(f"  Test set size:       {len(test_dataset)}")
        print(f"  Rounds:              {num_eval_rounds}")
        print(f"{'─'*60}")
        print(f"  Solve Rate:          {aggregate_summary['solve_rate']['mean']:.4f} "
              f"± {aggregate_summary['solve_rate']['std_dev']:.4f}")
        print(f"    Per round:         {aggregate_summary['solve_rate']['per_round']}")
        print(f"  Completion Acc:      {aggregate_summary['completion_accuracy']['mean']:.4f} "
              f"± {aggregate_summary['completion_accuracy']['std_dev']:.4f}")
        print(f"    Per round:         {aggregate_summary['completion_accuracy']['per_round']}")
        print(f"  Format Rate:         {aggregate_summary['format_rate']['mean']:.4f} "
              f"± {aggregate_summary['format_rate']['std_dev']:.4f}")
        print(f"  Generation Errors:   {aggregate_summary['num_errors']}")
        thinking_summary = aggregate_summary["avg_thinking_tokens"]
        if thinking_summary["mean"] is None:
            print("  Avg Thinking Tokens: N/A (no successful generation requests)")
        else:
            print(f"  Avg Thinking Tokens: {thinking_summary['mean']:.1f} "
                  f"± {thinking_summary['std_dev']:.1f}")
        print(f"{'='*60}")
        print(f"  Metrics saved to:    {metrics_path}")
        print(f"  Per-round responses: {responses_dir}/")

        # Save summary alongside the LoRA (only when a LoRA was evaluated)
        if lora_request is not None:
            summary_filename = f"eval_summary_{student_id}_{timestamp}.json"
            summary_path = os.path.join(lora_dir, summary_filename)
            with open(summary_path, "w") as f:
                json.dump({
                    "student_id": student_id,
                    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "results": aggregate_summary,
                }, f, indent=2)
            print(f"  Summary also saved to: {summary_path}")

**[Q2.2]** Run the multi-round evaluation. **[2 Marks]**

> ⚠️ **Note:**
> - This cell must evaluate the final LoRA adapter (or selected checkpoint) via `run_eval_for_targets()` on `gsm8k["test"]` using the parameters in `recipe_dic`.
> - Do **not** override `model.fast_generate` or replace `run_eval_for_targets()` with a custom evaluation loop in this cell. Violating any of the above compliance rules will result in **0 marks for Q2.2**. Evaluation results affected by the violation are invalid and will receive **no marks for the corresponding result-based criteria**.

In [ ]:
lora_dir = os.path.join(output_dir, "lora_eval_step3200")

for filename in ("adapter_config.json", "adapter_model.safetensors"):
    path = os.path.join(lora_dir, filename)
    if not os.path.isfile(path):
        raise FileNotFoundError(path)

recipe_dic["eval_config"]["eval_model"] = "both"
recipe_dic["eval_config"]["num_rounds"] = 4

print("LoRA checkpoint:", lora_dir)

run_eval_for_targets(
    eval_model=recipe_dic["eval_config"]["eval_model"],
    output_dir=output_dir,
    model=model,
    lora_dir=lora_dir,
    test_dataset=gsm8k["test"],
    system_prompt=system_prompt,
    recipe_dic=recipe_dic,
)

LoRA checkpoint: ./grpo_output_formal_run01/lora_eval_step3200
  Test set size: 1319
  Evaluating base model (lora_request=None).

Running 4 rounds of evaluation...

--- Evaluation Round 1/4 ---


Round 1: 100%|██████████| 1319/1319 [06:27<00:00,  3.40it/s]


  Round 1: solve_rate=0.0000 (0/1319), completion_acc=0.0000, format_rate=0.0000, avg_thinking_tokens=0.0, errors=0, time=388.0s

--- Evaluation Round 2/4 ---


Round 2: 100%|██████████| 1319/1319 [07:05<00:00,  3.10it/s]


  Round 2: solve_rate=0.0000 (0/1319), completion_acc=0.0000, format_rate=0.0000, avg_thinking_tokens=0.0, errors=0, time=425.9s

--- Evaluation Round 3/4 ---


Round 3: 100%|██████████| 1319/1319 [07:02<00:00,  3.12it/s]


... (4 outputs left out)

WARNING 11-25 15:23:57 [input_processor.py:141] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Round 1: 100%|██████████| 1319/1319 [09:14<00:00,  2.38it/s]


  Round 1: solve_rate=0.0000 (0/1319), completion_acc=0.0000, format_rate=0.0000, avg_thinking_tokens=0.0, errors=0, time=554.1s

--- Evaluation Round 2/4 ---


Round 2:  68%|██████▊   | 894/1319 [06:14<02:56,  2.41it/s]

# PART III: Analysis and Discussions
---

**[Q3.1]** What reasoning/training techniques have you applied in this project to enhance the reasoning capability of the Qwen2-1.5B-Instruct model on the GSM8k dataset? Based on the metrics result file you obtatined from the evaluations, how much does each technique contribute to the reasoning capability improvement? If you want to determine the contribution to performance improvement from each part, how you plan to conduct an ablation test? (Please learn what is an ablation test from GenAI/Search Engine. You don't need to implement the plan in this project). **[3 Marks]**



*Your answer here*

**[Q3.2]** Based on the training log `training_stats.csv` you obtatined from the evaluations, visualise how the loss and reward changes over training steps. What can you find from the chart? (Use GenAI tools to help you generate the visualisation codes.)  **[1 Mark]**



*Your answer here*

---
# Submission Instructions
---

Please follow the steps below to prepare and submit your project.

## Files to Include

Your submission must include the following files and folders:

| Item | Path |
|------|------|
| Base model evaluation responses | `grpo_output/responses_base/` (entire folder) |
| LoRA model evaluation responses | `grpo_output/responses_lora/` (entire folder) |
| Metrics JSON file(s) | `grpo_output/metrics_*.json` (all `.json` files in the root of `grpo_output/`) |
| Training log | `grpo_output/training_stats.csv` |
| Project notebook (one file) | `P02_Enhancing_Reasoning_Capability_in_LLMs_student.ipynb` (browser run) **or** `P02_Enhancing_Reasoning_Capability_in_LLMs_student_completed.ipynb` (`nbconvert` run) |

> ⚠️ **Important:** Before archiving, make sure your notebook has been fully executed and its outputs are visible. Missing training-cell output is acceptable when the automatically saved `grpo_output/training_stats.csv` is included in your submission. Evaluation outputs must be captured in the notebook and consistent with the submitted logs. Do **not** clear outputs before submission.


## How to Create the Archive

Name your archive file using your **QMUL Student ID** as the suffix:

```
submission_<YOUR_STUDENT_ID>.tar.gz
```

For example, if your student ID is `221154321`, your archive should be named:

```
submission_221154321.tar.gz
```

To create the archive, run the following command from the **root of the repository**:

The command below uses `P02_Enhancing_Reasoning_Capability_in_LLMs_student.ipynb` for a browser run. If you used `nbconvert`, replace the final filename with `P02_Enhancing_Reasoning_Capability_in_LLMs_student_completed.ipynb`. Include the notebook whose results match the logs you are submitting.

```bash
tar -czvf submission_<YOUR_STUDENT_ID>.tar.gz \
    grpo_output/responses_base/ \
    grpo_output/responses_lora/ \
    grpo_output/metrics_*.json \
    grpo_output/training_stats.csv \
    P02_Enhancing_Reasoning_Capability_in_LLMs_student.ipynb
```

## Upload

1. Download `submission_<YOUR_STUDENT_ID>.tar.gz` from the JHub server to your laptop.
2. Upload `submission_<YOUR_STUDENT_ID>.tar.gz` to the **Project Submission** link on Moodle before the deadline.

> ‼️ **Double-check your submission after uploading to make sure you upload the correct file.**
